# 7.28 Face recognition and metric learning

Face recognition is classification at enrollment time and geometry at verification time: compare normalized identities by angle, not by raw pixels.

Embeddings map crops into a space where identity is measured by cosine similarity, margins push impostors apart, and thresholds control false accepts and false rejects.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## The concept, built once

Verification uses normalized cosine similarity:

$$s(a,b)=\hat f_a^	op\hat f_b.$$

We verify $[3,4]	o[0.6,0.8]$, cosine $0.6$, margin penalty $0.04$, and a logit drop of $2$.

In [ ]:
def normalize(vec):
    vec = np.array(vec, dtype=float)
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm


def cosine(a, b):
    return float(np.dot(normalize(a), normalize(b)))


def margin_penalty(score, max_negative=0.4):
    violation = max(0.0, score - max_negative)
    return violation ** 2


def verify_embedding(a, b, threshold=0.65):
    score = cosine(a, b)
    return score, score >= threshold


u = np.array([3, 4], dtype=float)
unit = normalize(u)
score = cosine(unit, [1, 0])
angle = np.degrees(np.arccos(score))
penalty = margin_penalty(0.6, 0.4)
logit_drop = 10 * 0.8 - 10 * 0.6
assert np.allclose(unit, [0.6, 0.8])
assert round(np.linalg.norm(unit), 1) == 1.0
assert round(score, 1) == 0.6
assert round(angle, 2) == 53.13
assert round(penalty, 2) == 0.04
assert logit_drop == 2.0
print("normalized", unit)
print("cosine", score)
print("angle", round(angle, 2))
print("penalty", penalty)
print("logit drop", logit_drop)

## Dataset ladder

We use synthetic identities, not face downloads. Each rung creates genuine and impostor embedding pairs with rising noise, pose-like rotation, and norm variation. The metric is verification accuracy.

In [ ]:
def rotate(vec, degrees):
    theta = np.radians(degrees)
    rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return rot @ vec


def make_embedding_ladder():
    specs = [
        ("D1 clean identities", 2, 2, 0.02, 0),
        ("D2 more people", 3, 3, 0.05, 5),
        ("D3 noisy crops", 4, 4, 0.10, 10),
        ("D4 pose shift", 5, 5, 0.14, 18),
        ("D5 norm shift", 6, 6, 0.18, 25),
    ]
    rungs = []
    for seed, spec in enumerate(specs):
        name, identities, samples, noise, rotation = spec
        local_rng = np.random.default_rng(seed + 80)
        centers = []
        for i in range(identities):
            angle = 2 * np.pi * i / identities
            centers.append(np.array([np.cos(angle), np.sin(angle)]))
        embeddings = []
        labels = []
        for label, center in enumerate(centers):
            for j in range(samples):
                vec = rotate(center, rotation * j / max(1, samples - 1))
                vec = vec + local_rng.normal(0.0, noise, size=2)
                if name == "D5 norm shift":
                    vec = vec * (1.0 + 0.6 * j)
                embeddings.append(vec)
                labels.append(label)
        rungs.append({"name": name, "embeddings": np.array(embeddings), "labels": np.array(labels)})
    return rungs


def verification_pairs(embeddings, labels):
    pairs = []
    targets = []
    for i in range(len(embeddings) - 1):
        for j in range(i + 1, len(embeddings)):
            pairs.append((embeddings[i], embeddings[j]))
            targets.append(labels[i] == labels[j])
    return pairs, np.array(targets)


def verification_accuracy(embeddings, labels, threshold=0.65, use_norm=True):
    pairs, targets = verification_pairs(embeddings, labels)
    predictions = []
    scores = []
    for a, b in pairs:
        if use_norm:
            score = cosine(a, b)
        else:
            score = float(np.dot(a, b))
        scores.append(score)
        predictions.append(score >= threshold)
    return float(np.mean(np.array(predictions) == targets)), np.array(scores), targets


embed_rungs = make_embedding_ladder()
for rung in embed_rungs:
    print(rung["name"], "embeddings", rung["embeddings"].shape, "identities", len(set(rung["labels"].tolist())))

fig, axes = plt.subplots(1, 5, figsize=(12, 2.4))
for ax, rung in zip(axes, embed_rungs):
    emb = np.array([normalize(v) for v in rung["embeddings"]])
    ax.scatter(emb[:, 0], emb[:, 1], c=rung["labels"])
    ax.set_title(rung["name"])
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
plt.tight_layout()

In [ ]:
embed_results = []
for rung in embed_rungs:
    acc, scores, targets = verification_accuracy(rung["embeddings"], rung["labels"])
    embed_results.append({"name": rung["name"], "acc": acc, "scores": scores, "targets": targets})

print("rung                 verification accuracy")
for result in embed_results:
    print(f"{result['name']:<20} {result['acc']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, rung in enumerate(embed_rungs):
    emb = np.array([normalize(v) for v in rung["embeddings"]])
    axes[0, i].scatter(emb[:, 0], emb[:, 1], c=rung["labels"])
    axes[0, i].set_title(rung["name"])
    axes[0, i].set_xlim(-1.2, 1.2)
    axes[0, i].set_ylim(-1.2, 1.2)

scores = [item["acc"] for item in embed_results]
axes[1, 0].plot(range(1, 6), scores, marker="o")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("verification accuracy")
axes[1, 0].set_title("metric learning ladder")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: raw dot products confuse norm with identity

Unnormalized embeddings can create false matches because large vector length inflates dot products. L2 normalization and threshold calibration fix the score scale.

In [ ]:
hard = embed_rungs[-1]
raw_acc, raw_scores, targets = verification_accuracy(hard["embeddings"], hard["labels"], threshold=0.65, use_norm=False)
norm_acc, norm_scores, targets = verification_accuracy(hard["embeddings"], hard["labels"], threshold=0.65, use_norm=True)
thresholds = np.linspace(0.1, 0.9, 9)
calibrated = []
for threshold in thresholds:
    acc, scores, target_values = verification_accuracy(hard["embeddings"], hard["labels"], threshold=threshold, use_norm=True)
    calibrated.append(acc)
best_index = int(np.argmax(calibrated))
print("raw-dot accuracy", round(raw_acc, 2))
print("normalized accuracy", round(norm_acc, 2))
print("best calibrated threshold", round(float(thresholds[best_index]), 2), "accuracy", round(float(calibrated[best_index]), 2))

## Evaluate it + Practice

- Metric: verification accuracy over genuine and impostor pairs; no-skill baseline predicts every pair is different.
- Sanity check: normalized vectors have unit norm before cosine comparison.
- Ablation: use raw dot products and D5 norm shift creates false matches.
- Failure signal: a threshold calibrated on easy rungs fails under pose or norm shift.

Practice:
1. Add a third embedding dimension.
2. Sweep thresholds and plot false accepts.
3. Increase the margin and inspect impostor penalties.